In [ ]:
schools_pts.explore()

In [ ]:
from routingpy import Valhalla
from pprint import pprint
import shapely
import contextily as cx
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import pandas as pd

# CATCHMENT_AREAS = [5, 10, 15, 20, 25, 30]
CATCHMENT_AREAS = [5, 10, 15, 20]

# MODES_OF_TRANSPORTATION = ['pedestrian']
MODES_OF_TRANSPORTATION = ['bicycle']
# MODES_OF_TRANSPORTATION = ['pedestrian', 'bicycle', 'auto']
# MODES_OF_TRANSPORTATION = ['pedestrian', 'bicycle']

def calculate_isochrones(latlons, catchment_areas = CATCHMENT_AREAS):
    MAX_INTERVALS = 4 # Valhalla API limit: maximum number of intervals per request
    
    client = Valhalla(retry_over_query_limit=True)

    # Nested dictionary structure:
    # {mode_of_transport: {time_interval: [list_of_polygons]}}
    isochrones_by_trans_by_time = {}

    # Split catchment areas into chunks of size <= MAX_INTERVALS
    # Example: [5,10,15,20,25,30] → [[5,10,15,20],[25,30]]
    chunked_catchments_areas = [
        catchment_areas[i:i + MAX_INTERVALS]
        for i in range(0, len(catchment_areas), MAX_INTERVALS)
    ]
    
    for chunked_catchment_area in chunked_catchments_areas:
        for latlon in latlons:
            for mode_of_transport in MODES_OF_TRANSPORTATION:
                # Convert minutes → seconds because Valhalla expects seconds
                # Example: [5,10] → [300,600]
                isochrones = client.isochrones(
                    locations=latlon,
                    profile=mode_of_transport,
                    intervals=[c * 60 for c in chunked_catchment_area],
                    interval_type='time',
                    polygons=True
                )
        
                for iso in isochrones:
                        isochrones_by_trans_by_time.setdefault(mode_of_transport, {}).setdefault(iso.interval, []).append(Polygon(iso.geometry[0])) # iso.geometry[0] extracts the outer ring of the polygon
                    
    # Merge (union) polygons that share the same mode and time threshold
    # This combines overlapping isochrones from multiple origins
    for mode_of_transport, isochrones_by_time in isochrones_by_trans_by_time.items():
        for time, polygons in isochrones_by_time.items():
            isochrones_by_trans_by_time[mode_of_transport][time] = shapely.union_all(polygons)

    # Flatten nested dictionary into tabular format
    # Each row represents one (mode, time, geometry)
    df = pd.DataFrame(
        [
            (mode_of_transport, time, polygon)
            for mode_of_transport, isochrones_by_time in isochrones_by_trans_by_time.items()
            for time, polygon in isochrones_by_time.items()
        ],
        columns=["mode_of_transportation", "time", "geometry"]
    )

    final_isochrones = gpd.GeoDataFrame(df, crs="EPSG:4326").sort_values(by="time", ascending=False)
    
    return final_isochrones

In [ ]:
latlons = list(
    zip(schools_pts.geometry.x,
        schools_pts.geometry.y)
)

In [ ]:
isochrones = calculate_isochrones(latlons)
isochrones.sort_values("time", ascending=False)

In [ ]:
def save_isochrones(isochrones_gdf, city_name):
    for _, row in isochrones_gdf.iterrows():
        one = isochrones_gdf.loc[[row.name]]
    
        layer_name = f"mode_{row['mode_of_transportation']}_time_{int(row['time'])}"
        
        one.to_file(
            f'output/{city_name}_isochrones.gpkg',
            layer=layer_name,
            driver="GPKG"
        )

save_isochrones(isochrones, 'espoo')

In [ ]:
isochrones.head()

In [ ]:
isochrones.explore()

In [ ]:
espoo = ox.geocode_to_gdf("Espoo, Finland")
espoo = espoo.to_crs(4326)

m = espoo.explore(
    tiles="CartoDB positron",
    color="black",
    style_kwds={"fillColor": "lightblue", "fillOpacity": 0.2, "weight": 2},
    name="Espoo boundary",
)

m = isochrones.explore(
    m=m,
    column="time",
    cmap="YlOrRd",
    legend=True,
    style_kwds={"fillOpacity": 0.35, "weight": 1},
    tooltip=["time"],
    name="Isochrones",
)

m = pts.explore(
    m=m,
    color="blue",
    marker_kwds={"radius": 2},
    tooltip=["name"],
    name="School",
)

m

In [ ]:
def calculate_population_catchments(pop_gdf, isochrones):
    rows = []
    
    total_population = pop_gdf.population_sum.sum()

    for trans_mode in MODES_OF_TRANSPORTATION:
        isochrones_for_mode = isochrones[isochrones['mode_of_transportation'] == trans_mode]
        for _, iso in isochrones_for_mode.iterrows():
            pop_cells_in_isochrone = pop_gdf[pop_gdf.intersects(iso.geometry)]
            pop_in_iso = pop_cells_in_isochrone.population_sum.sum()
            
            rows.append({
                "mode_of_transportation": trans_mode,
                "time": iso.time,
                "time_min": iso.time / 60,
                "population_share": pop_in_iso / total_population
            })
    
    population_catchment = (
        pd.DataFrame(rows)
        .sort_values(["mode_of_transportation", "time"])
        .reset_index(drop=True)
    )

    return population_catchment

In [ ]:
import geopandas as gpd
from pathlib import Path

fp = Path(r"output\espoo_children_population.geojson")

espoo_pop = gpd.read_file(fp)

espoo_pop = espoo_pop.to_crs(4326)

espoo_pop.head()

In [ ]:
espoo_child_population_catchment = calculate_population_catchments(espoo_pop, isochrones)

In [ ]:
espoo_child_population_catchment.to_csv(
    "output/espoo_child_population_catchment.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('bmh')

fig, ax = plt.subplots(figsize=(8,4))

# Plot each line and give it a label
for (mode), g in espoo_child_population_catchment.groupby(["mode_of_transportation"]):
    ax.plot(
        g["time_min"],
        g["population_share"],
        marker="o",
        markersize=5,
        label=f"{mode}"  # ← add a label so legend shows
    )

# Vertical line at 15 minutes
ax.axvline(x=15, color='black', linestyle='--', linewidth=0.8)

# "15 minutes" label with transparent grey box
ax.text(
    x=15 + 1, 
    y=ax.get_ylim()[0]+0.2, 
    s="15 minutes", 
    color='black', 
    rotation=0, 
    va='top', 
    ha='left',
    bbox=dict(facecolor='lightgrey', alpha=0.5, edgecolor='none', boxstyle='round,pad=0.3')
)

ax.set_xlabel("Travel time (minutes)")
ax.set_ylabel("Population share (%)")
ax.set_title("Espoo", color='black')

# Show legend
legend = ax.legend()
plt.setp(legend.get_texts(), color='black')
plt.setp(legend.get_title(), color='black')

fig.savefig(f"output/child_population_accessibility_potential_espoo.png", dpi=300, bbox_inches='tight', facecolor='#e7e7e7')

plt.show()
